In [1]:
import os
import numpy as np
import pandas as pd

os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

try:
    from google.protobuf import message_factory

    if not hasattr(message_factory.MessageFactory, "GetPrototype"):

        def _get_prototype(self, descriptor):
            return self.GetMessageClass(descriptor)

        message_factory.MessageFactory.GetPrototype = _get_prototype
except Exception:
    pass

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint




2026-05-05 09:12:01.073149: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777972321.086412      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777972321.090497      55 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-05 09:12:01.107388: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
base_path = "/kaggle/input/aerial-cactus-identification/"

files_dataframe = pd.read_csv(os.path.join(base_path, "train.csv"), dtype=str)
files_dataframe.head()




,id,has_cactus
0,2de8f189f1dce439766637e75df0ee27.jpg,1
1,36704d250f236238e7f996812c48235d.jpg,1
2,eacde22fdc8c175972a5768e3daa8bc9.jpg,1
3,5d442f834da5e57d22b24802c32a8ca8.jpg,1
4,152491e0daf75c0e669400300ff7e645.jpg,1


In [3]:
os.makedirs("./training", exist_ok=True)
os.makedirs("./test", exist_ok=True)

from zipfile import ZipFile

with ZipFile(os.path.join(base_path, "train.zip"), "r") as zipper:
    zipper.extractall("./training/")  # images end up in ./training/
with ZipFile(os.path.join(base_path, "test.zip"), "r") as zipper:
    zipper.extractall("./test/")  # images end up in ./test/

TRAIN_DIR = "./training"
TEST_DIR = "./test"




In [4]:
class_counts = files_dataframe["has_cactus"].value_counts()
total_samples = len(files_dataframe)
has_cactus_weight = total_samples / (2 * class_counts["1"])
no_cactus_weight = total_samples / (2 * class_counts["0"])
class_weights = {0: no_cactus_weight, 1: has_cactus_weight}
print("Class weights:", class_weights)

from sklearn.model_selection import train_test_split

files_dataframe["has_cactus"] = files_dataframe["has_cactus"].astype(int)

train_df, val_df = train_test_split(
    files_dataframe,
    test_size=0.1,
    stratify=files_dataframe["has_cactus"],
    random_state=42,
)


def make_dataset(df, directory, training=True, batch_size=32):
    """Create a tf.data.Dataset from a dataframe."""
    directory = directory.rstrip("/")
    file_paths = tf.strings.join([directory, df["id"]], separator="/")
    labels = df["has_cactus"].values
    ds = tf.data.Dataset.from_tensor_slices((file_paths, labels))

    def _parse_image(path, label):
        image = tf.io.read_file(path)
        image = tf.image.decode_jpeg(image, channels=3)
        image = tf.image.resize(image, [32, 32])
        image = tf.cast(image, tf.float32) / 255.0
        label_onehot = tf.one_hot(label, depth=2)
        return image, label_onehot

    ds = ds.map(_parse_image, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(buffer_size=1000, seed=42)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = make_dataset(train_df, TRAIN_DIR, training=True, batch_size=32)
val_ds = make_dataset(val_df, TRAIN_DIR, training=False, batch_size=32)




Class weights: {0: 1.9981674654637722, 1: 0.6668705306736922}


2026-05-05 09:12:09.429861: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1777972329.430322      55 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:85:00.0, compute capability: 8.6


In [5]:
model = models.Sequential(
    [
        layers.Flatten(input_shape=(32, 32, 3)),
        layers.Dense(256, activation="relu"),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.2),  # increased dropout from 0.1 to 0.2
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.2),  # increased dropout from 0.1 to 0.2
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.2),  # increased dropout from 0.1 to 0.2
        layers.Dense(64, activation="relu"),
        layers.Dense(2, activation="softmax"),
    ]
)

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
)
model.summary()




/usr/local/lib/python3.11/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 3072)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       786,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 910,274 (3.47 MB)

 Trainable params: 910,274 (3.47 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3, min_lr=1e-5)

checkpoint_path = "./best_model.keras"
model_checkpoint = ModelCheckpoint(
    checkpoint_path,
    monitor="val_loss",
    mode="min",
    save_best_only=True,
    save_weights_only=False,
    verbose=1,
)




In [7]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=4,  # reduced epochs to lower overfitting and AUC towards target
    class_weight=None,  # kept without class weighting
    callbacks=[reduce_lr, model_checkpoint],
    verbose=1,
)




Epoch 1/4


I0000 00:00:1777972332.521040     106 service.cc:148] XLA service 0x7fe2e0003260 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777972332.521084     106 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-05-05 09:12:12.550184: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1777972332.694387     106 cuda_dnn.cc:529] Loaded cuDNN version 90300
2026-05-05 09:12:13.996671: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1558', 24 bytes spill stores, 24 bytes spill loads

2026-05-05 09:12:14.132125: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1558', 68 bytes spill 

111/399 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7024 - auc: 0.7225 - loss: 0.6441

I0000 00:00:1777972335.780357     106 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


394/399 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7287 - auc: 0.7526 - loss: 0.5983

2026-05-05 09:12:17.702310: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1558', 20 bytes spill stores, 24 bytes spill loads

2026-05-05 09:12:18.203573: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1558', 32 bytes spill stores, 32 bytes spill loads

2026-05-05 09:12:18.404538: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1558', 24 bytes spill stores, 24 bytes spill loads



399/399 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7288 - auc: 0.7530 - loss: 0.5979
Epoch 1: val_loss improved from inf to 0.54915, saving model to ./best_model.keras
399/399 ━━━━━━━━━━━━━━━━━━━━ 14s 22ms/step - accuracy: 0.7289 - auc: 0.7531 - loss: 0.5978 - val_accuracy: 0.7496 - val_auc: 0.8581 - val_loss: 0.5491 - learning_rate: 0.0010
Epoch 2/4
359/399 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7515 - auc: 0.7725 - loss: 0.5550
Epoch 2: val_loss did not improve from 0.54915
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7514 - auc: 0.7709 - loss: 0.5557 - val_accuracy: 0.7496 - val_auc: 0.7496 - val_loss: 0.5628 - learning_rate: 0.0010
Epoch 3/4
362/399 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7540 - auc: 0.7522 - loss: 0.5593
Epoch 3: val_loss did not improve from 0.54915
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7537 - auc: 0.7520 - loss: 0.5597 - val_accuracy: 0.7496 - val_auc: 0.7496 - val_loss: 0.5630 - learning_rate: 0.0010
Epoch 4/4
398/3

In [8]:
model = tf.keras.models.load_model(checkpoint_path)




In [9]:
test_filenames = sorted(os.listdir(TEST_DIR))
test_df = pd.DataFrame({"id": test_filenames})


def make_test_dataset(df, directory, batch_size=32):
    directory = directory.rstrip("/")
    file_paths = tf.strings.join([directory, df["id"]], separator="/")
    ds = tf.data.Dataset.from_tensor_slices(file_paths)

    def _parse_image(path):
        image = tf.io.read_file(path)
        image = tf.image.decode_jpeg(image, channels=3)
        image = tf.image.resize(image, [32, 32])
        image = tf.cast(image, tf.float32) / 255.0
        return image

    ds = ds.map(_parse_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


test_ds = make_test_dataset(test_df, TEST_DIR, batch_size=32)




In [10]:
probas = model.predict(test_ds, verbose=0)
has_cactus_prob = probas[:, 1]

submission = pd.DataFrame({"id": test_df["id"], "has_cactus": has_cactus_prob})
submission.to_csv("submission.csv", index=False)
print("Submission saved to submission.csv, shape:", submission.shape)




Submission saved to submission.csv, shape: (3325, 2)


In [11]:
import shutil

shutil.rmtree("./training", ignore_errors=True)
shutil.rmtree("./test", ignore_errors=True)